In [ ]:
import math
import random
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from time import sleep
from collections import deque
from itertools import count

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from importnb import Notebook
with Notebook():
    from LabLatencyModel import LatencyModel
    from LabCacheEngine import CacheEngineEnv
    from LabUserTileRequest import UserTileRequestEvents
    from LabEnvWrapper import EnvWrapper

import sys
sys.path.append(r'c:\Users\es25591\Workspace\CacheVideoPredict360\Sources')
# sys.path.append('/home/eduardo/Workspace/CacheVideoPredict360/Sources')
from Common.Utils import save_training_results


In [ ]:
class ReplayMemory:
    def __init__(self, capacity):
        self.memory = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        # state shape: (seq_len, input_size)
        self.memory.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

class LSTM_DQN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(LSTM_DQN, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM Layer: Takes sequence of viewports
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        
        # Fully Connected Layer: Maps LSTM output to Q-values (one per tile index)
        self.fc = nn.Linear(hidden_size, output_size)

        self.sigmoid = nn.Sigmoid() # Squish output between 0 and 1
        
    def forward(self, x):
        # x: (Batch, Seq_Len, Features)
        lstm_out, _ = self.lstm(x)
        
        # We take the last time step's output
        last_step_out = lstm_out[:, -1, :]
        
        # Produce logits then probabilities
        logits = self.fc(last_step_out)
        probs = self.sigmoid(logits)

        return probs, logits


# --- New Probability Agent ---
class ProbabilityAgent:
    def __init__(
        self, 
        n_tiles, 
        seq_len, 
        threshold=0.5, 
        learning_rate=1e-3,
        batch_size=32,
        capacity=10000,
        epsilon_start=1.0,
        epsilon_min=0.05,
        epsilon_decay=0.995
    ):
        self.n_tiles = n_tiles
        self.seq_len = seq_len
        self.threshold = threshold
        self.epsilon = epsilon_start
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        
        # Replay Memory
        self.memory = deque(maxlen=capacity)
        self.batch_size = batch_size
        
        # Network
        self.model = LSTM_DQN(n_tiles, 128, n_tiles)
        self.optimizer = optim.Adam(
            self.model.parameters(), 
            lr=learning_rate
        )
        self.criterion = nn.BCELoss() # Binary Cross Entropy Loss
        
    def predict_viewport(self, state_seq):
        """
        Input: state_seq (Seq_Len, n_tiles)
        Output: Binary mask (n_tiles,) based on threshold
        """
        # Convert to tensor (Batch=1)
        state_tensor = torch.FloatTensor(state_seq).unsqueeze(0)
        
        with torch.no_grad():
            probs, _ = self.model(state_tensor) # Shape: (1, n_tiles)
            
        # Convert to numpy and apply threshold
        probs_np = probs.squeeze(0).numpy()
        
        # Action: 1 if prob > threshold, else 0
        mask = (probs_np > self.threshold).astype(int)
        return mask
    
    def remember(self, state, actual_future_mask):
        """
        Store the state and the ACTUAL mask that happened (Ground Truth).
        We don't need 'reward' for training in this supervised setup, 
        because 'perfect prediction' implies 'max reward'.
        """
        self.memory.append((state, actual_future_mask))
        
    def train_step(self):
        if len(self.memory) < self.batch_size:
            return 0.0
        
        batch = random.sample(self.memory, self.batch_size)
        states, targets = zip(*batch)
        
        # Convert to tensors
        states_tensor = torch.FloatTensor(np.array(states))   # (B, Seq, Input)
        targets_tensor = torch.FloatTensor(np.array(targets)) # (B, Input)
        
        # Forward pass
        probs, _ = self.model(states_tensor)  # (B, Input)
        
        # Compute loss
        loss = self.criterion(probs, targets_tensor)
        
        # Backpropagation
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        return loss.item()
    
    def update_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

In [ ]:
if __name__ == "__main__":
    n_episodes = 200
    n_users = 100
    arrival_rate = 5.0  # users per second
    alpha = 1.0
    step_size = 5.0
    max_users_capacity = n_users
    n_videos = 100
    n_gops = 60
    n_layers = 2
    n = 4
    n_tiles = n * n
    max_capacity = 100e6  # 100 MB

    # Hyperparameters for RL
    epsilon_start = 1.0
    epsilon_min = 0.05
    epsilon_decay = 0.99
    gamma = 0.99
    learning_rate = 1e-3
    batch_size = 64
    capacity = 10000
    window_len = 5  # LSTM sequence length (history window)
    caching_threshold = 0.4 # If P(tile) > 40%, cache it

    # CPT parameters
    theta = 0.5
    lam = 3.7183

    # Initialize User Environment
    users_env = UserTileRequestEvents(
        n_users=n_users,
        n_videos=n_videos,
        n_gops=n_gops,
        n_layers=n_layers,
        n_tiles=n_tiles,
        alpha=alpha,
        n=n,
        users_viewport_tiles=None,
        requested_videos=None,
        users_arrivals=None,
        arrival_rate=arrival_rate
    )

    cache_env = CacheEngineEnv(
        n_users=n_users,
        n_videos=n_videos,
        n_layers=n_layers,
        n_tiles=n_tiles,
        n_gops=n_gops,
        cache_capacity=max_capacity,
    )

    # Create latency model (replace numbers with your real config)
    P = 1; max_U = n_users
    lat_model = LatencyModel(
        P=P, 
        max_U=max_U,
        R_M_D=80e6,   # 640 Mbps -> 80e6 B/s 
        R_C_M=1.25e9, # 10 Gbps -> 1.25e9 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 40e6, dtype=float),       # 320 Mbps -> 40e6 B/s
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float),    # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,    # 1 ms
        mec_fixed_delay=0.005,   # 5 ms
        cloud_fixed_delay=0.05   # 50 ms
    )

    env = EnvWrapper(
        n=n,
        n_layers=n_layers,
        users_env=users_env, 
        cache_env=cache_env, 
        latency_model=lat_model,
        theta=theta,
        lam=lam
    )

    print(
        f"Experiment ===================\n"
        f"Total users: {n_users}\n"
        f"Cache capacity: {max_capacity}MB\n"
        f"Video matrix: {n_videos}x{n_layers}x{n*n}\n"
        f"==============================\n"
    )
    
    agent = ProbabilityAgent(
        n_tiles=n_tiles, 
        seq_len=window_len, 
        threshold=caching_threshold,
        learning_rate=learning_rate,
        batch_size=batch_size,
        capacity=capacity,
        epsilon_start=epsilon_start,
        epsilon_min=epsilon_min,
        epsilon_decay=epsilon_decay
    )

    results = []
    scores = []
    for ep in range(n_episodes):

        obs, info = env.reset()
        
        user_histories = {
            u: deque(
                [np.zeros(n*n) for _ in range(window_len)], maxlen=window_len
            ) for u in range(max_users_capacity)
        }

        cache_hits = 0
        cache_misses = 0
        enhanced_layer_cache_hits = 0
        enhanced_layer_cache_misses = 0
        base_layer_cache_hits = 0
        base_layer_cache_misses = 0
        
        total_reward = 0.0

        for step in count():

            # --- AGENT ACTIONS ---
            reqs_state = info['users_requests']
            active_users = [ 
                (req['u'], req['video'], req['gop']) for req in reqs_state if req['gop'] < n_gops
            ]

            actions = []

            for u, video, gop in active_users:
                state_seq = np.array(user_histories[u])

                # Predict mask
                predicted_viewport = agent.predict_viewport(state_seq)

                actions.append({
                    'user': u,
                    'video': video,
                    'tiles': predicted_viewport,
                    'gop': gop
                })

            # --- STEP ENV ---
            obs, rewards, done, info = env.step(actions)

            total_reward += float(rewards)
            cache_hits += info["cache_hits"]
            cache_misses += info["cache_misses"]
            enhanced_layer_cache_hits += info["enh_layer_cache_hits"]
            enhanced_layer_cache_misses += info["enh_layer_cache_misses"]
            base_layer_cache_hits += info["base_layer_cache_hits"]
            base_layer_cache_misses += info["base_layer_cache_misses"]
            

            # --- TRAINING / HISTORY UPDATE ---
            reqs_next_state = info['users_requests']

            # --- Training Update ---
            for u, video, gop in active_users:
                state_seq = np.array(user_histories[u])

                # Actual future mask based on what was requested
                actual_viewport = np.zeros(n_tiles, dtype=int)
                target_indices = [
                    tile['tile']
                    for req in reqs_next_state 
                    if req['u'] == u
                    for tile in req['tiles']
                    if tile['layer'] == 1
                ]
                actual_viewport[target_indices] = 1
                
                # Update user history with the actual observed viewport (next state)
                user_histories[u].append(actual_viewport)

                # Store in replay memory
                agent.remember(state_seq, actual_viewport)

            # Train the agent (once per step, or multiple times)
            agent.train_step()

            # print(f"Step {step}, Active Users: {len(active_users)}")
            # print(f"Request State: {reqs_state}")
            # print(f"Action: {actions}")
            # print(f"Next Request State: {reqs_next_state}")
            # print(
            #     f"Reward: {rewards}, "
            #     f"Cache Hits: {info['cache_hits']}, "
            #     f"Cache Misses: {info['cache_misses']}"
            # )
            # print("-----")

            if done:
                break
            
        agent.update_epsilon()

        print(
            f"Episode {ep+1}/{n_episodes}, "
            f"Total Reward: {total_reward:.4f}, "
            f"cache_hits: {cache_hits}, "
            f"cache_misses: {cache_misses}, "
            f"Epsilon: {agent.epsilon:.4f}"
        )

        filename = (
            f"probs_predictor_"
            f"U{n_users}_V{n_videos}_G{n_gops}_L{n_layers}_N{n}_"
            f"cap{int(max_capacity/1e6)}MB_eps{epsilon_min}_"
            f"thr{int(caching_threshold*100)}pct.csv"
        )
        save_training_results(
            path_='/home/eduardo/Workspace/CacheVideoPredict360/Results',
            filename=filename,
            ep=ep,
            total_reward=total_reward,
            cache_hits=cache_hits,
            cache_misses=cache_misses,
            enhanced_layer_cache_hits=enhanced_layer_cache_hits,
            enhanced_layer_cache_misses=enhanced_layer_cache_misses,
            base_layer_cache_hits=base_layer_cache_hits,
            base_layer_cache_misses=base_layer_cache_misses,
            agent=agent
        )

        scores.append(total_reward)

        results.append({
            "episode": ep,
            "total_reward": total_reward,
            "cache_hits": cache_hits,
            "cache_misses": cache_misses,
        })

In [ ]:
episodes = range(1, len(scores) + 1)
cache_hits_series = [r["cache_hits"] for r in results]
cache_misses_series = [r["cache_misses"] for r in results]

fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharex=True)

# Rewards with moving average
axes[0].plot(episodes, scores, label="Episode reward", alpha=0.7, color="blue")
w = max(1, min(20, len(scores) // 10))
if w > 1:
    ma = [sum(scores[i - w:i]) / w for i in range(w, len(scores) + 1)]
    axes[0].plot(range(w, len(scores) + 1), ma, label=f"Moving avg (w={w})", color="orange")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Total reward")
axes[0].set_title("Training Rewards")
axes[0].set_ylim(bottom=0)
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Cache hits
axes[1].plot(episodes, cache_hits_series, label="Cache hits", color="green", alpha=0.8)
axes[1].set_title("Cache Hits per Episode")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Hits")
axes[1].set_ylim(bottom=0)
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# Cache misses
axes[2].plot(episodes, cache_misses_series, label="Cache misses", color="red", alpha=0.8)
axes[2].set_title("Cache Misses per Episode")
axes[2].set_xlabel("Episode")
axes[2].set_ylabel("Misses")
axes[2].set_ylim(bottom=0)
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()